# ML-04 — Search Intelligence Data Contract

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Unit of analysis + time window

*One row = one what, over which dates? State it, then verify it below.*

One row is one content item for one client, aggregated across all days in March 2026 (month=2026-03). Raw data comes from fact_content_daily_performance, which has one row per page per client per day. We aggregate to remove the daily dimension.

The table I use is fact_content_daily_performance, partitioned by month=2026

The time window will be March 2026 (2026-03-01 to 2026-03-31)

In [4]:
import os
import getpass
import duckdb

# Get your Hugging Face token safely (never paste directly)
HF_TOKEN = os.environ.get('HF_TOKEN') or getpass.getpass('Paste your HF token (hf_...): ')

# Connect DuckDB to Hugging Face
con = duckdb.connect()
con.execute(f"CREATE OR REPLACE SECRET hf (TYPE huggingface, TOKEN '{HF_TOKEN}')")

# Define table paths
REL = 'hf://datasets/FlyRank/internship-warehouse'
TABLES = {
    'fact_daily': f"read_parquet('{REL}/fact_content_daily_performance/**/*.parquet')",
    'dim_content': f"read_parquet('{REL}/dim_content.parquet')",
    'dim_clients': f"read_parquet('{REL}/dim_clients.parquet')",
}

# Verify connection
for name, src in TABLES.items():
    n = con.sql(f'SELECT COUNT(*) FROM {src}').fetchone()[0]
    print(f'{name:20} {n:>12,} rows')

fact_daily             78,835,655 rows
dim_content               519,606 rows
dim_clients                   104 rows


## 2. Fields: feature / label / context / excluded

*Sort every field you plan to touch into these four buckets. Excluded needs a why.*

In [5]:
# You'll aggregate these across all days in 2026-03
fact_daily_columns = [
    'report_date',           # Context or Excluded?
    'month',                 # Context or Excluded?
    'client_hash_id',        # Context
    'content_hash_id',       # Context
    'gsc_data_available',    # Context (filter flag)
    'ga4_data_available',    # Context (filter flag)
    'gsc_impressions',       # Feature
    'gsc_clicks',            # Excluded (why?)
    'gsc_sum_position',      # Feature
    'gsc_avg_position',      # Feature
    'ga4_pageviews',         # Excluded (why?)
    'ga4_sessions',          # Excluded (why?)
    'ga4_engaged_sessions',  # Excluded (why?)
    'scroll_events',         # Excluded (why?)
    'sessions_ai',           # Excluded (why?)
    'ai_chatgpt',            # Excluded (why?)
    # ... (all other GA4 and AI columns)
]

In [6]:
dim_content_columns = [
    'content_hash_id',       # Context
    'client_hash_id',        # Context
    'keyword_hash_id',       # Context or Excluded?
    'url_hash_id',           # Context or Excluded?
    'word_count',            # Feature
    'char_count',            # Feature or Excluded? (why?)
    'content_created_date',  # Feature (derive to days_since_creation)
    'content_updated_date',  # Feature or Excluded?
    'content_type',          # Feature or Context?
    'search_volume',         # Feature or Excluded?
    'competition',           # Feature
    'competition_level',     # Feature
    'cpc',                   # Feature or Excluded?
    'main_intent',           # Feature or Context?
    'backlinks',             # Feature or Excluded?
    'category_count',        # Feature or Excluded?
    'last_optimized_date',   # Feature (derive to days_since_optimization)
    'is_published',          # Context (filter flag)
    'is_deleted',            # Context (filter flag)
    'provider_used',         # Context or Excluded?
    'model_used',            # Context or Excluded?
]

## 2. Fields: Feature / Label / Context / Excluded

### Features (knowable before decision)
| Column             | Table | Why |
|--------------------|-------|-----|
| gsc_impressions    | fact_daily | Visibility; doesn't depend on clicks |
| gsc_avg_position   | fact_daily | Ranking assigned by Google before clicks |
| word_count         | dim_content | Content length; independent of CTR |
| competition_level  | dim_content | Keyword difficulty; external signal |
| has_been_optimized | dim_content | Freshness; knowable at decision time |

### Label / Proxy
| Derived From | Definition |
|--------------|-----------|
| gsc_clicks, gsc_impressions, gsc_avg_position | CTR is >1 std dev below position_tier mean, with 100+ impressions |

### Context (grouping, filtering, not model inputs)
| Column | Table | Why |
|--------|-------|-----|
| content_hash_id | fact_daily | Join key; identifies the page |
| client_hash_id | fact_daily | Join key; identifies the client |
| report_date | fact_daily | Time filter; not a feature |
| gsc_data_available | fact_daily | Filter flag; shows data quality |
| is_published | dim_content | Filter flag; exclude drafts |

### Excluded
| Column               | Table | Why |
|----------------------|-------|-----|
| gsc_clicks           | fact_daily | Used to compute CTR label; would leak |
| ctr                  | fact_daily | IS the label source; circular |
| ga4_pageviews        | fact_daily | Downstream of clicks; outcome data |
| scroll_events        | fact_daily | Requires click first; outcome data |
| sessions_ai          | fact_daily | Sessions require clicks; outcome |
| *any ai sessions*    | fact_daily | Subset of sessions; outcome data |
| char_count           | dim_content | Multicollinear with word_count |
| content_updated_date | dim_content | Could be future-dated after decision |

## 3. Verify it with queries (grain, counts, missing values, windows)

*Every claim above gets a query cell here. A contract claim without a query next to it is a guess.*

In [15]:
print("=" * 60)
print("BUILDING FEATURE FRAME — Step 1: Aggregate fact_daily")
print("=" * 60)

daily_agg = con.sql(f"""
  SELECT
    client_hash_id,
    content_hash_id,
    SUM(gsc_impressions) as total_impressions,
    AVG(gsc_avg_position) as avg_position_march,
    COUNT(DISTINCT report_date) as days_with_data
  FROM {TABLES['fact_daily']}
  WHERE month = '2026-03'
    AND gsc_data_available = TRUE
  GROUP BY client_hash_id, content_hash_id
""").df()

print(f"Aggregated rows: {len(daily_agg):,}")
print(daily_agg.head())

BUILDING FEATURE FRAME — Step 1: Aggregate fact_daily
Aggregated rows: 176,738
            client_hash_id           content_hash_id  total_impressions  \
0  client_73cda7b4e4f265ea  content_1e392a54ca96730d              207.0   
1  client_73cda7b4e4f265ea  content_2e7b4f25a1033e4b              165.0   
2  client_73cda7b4e4f265ea  content_2bbf031b7abf488a              177.0   
3  client_73cda7b4e4f265ea  content_368a3b619849acb0            10704.0   
4  client_73cda7b4e4f265ea  content_4dd3e1ea3eab7fd0             1154.0   

   avg_position_march  days_with_data  
0           33.675364              31  
1           51.321032              30  
2           49.907002              31  
3            5.504164              31  
4            9.736655              31  


In [18]:
len(daily_agg)

176738

In [20]:
print("=" * 60)
print("Step 2 (revised): Pull dim_content features with optimization flag")
print("=" * 60)

content_features = con.sql(f"""
  SELECT
    client_hash_id,
    content_hash_id,
    word_count,
    competition_level,
    CASE WHEN last_optimized_date IS NOT NULL THEN TRUE ELSE FALSE END as has_been_optimized,
    is_published
  FROM {TABLES['dim_content']}
  WHERE is_published = TRUE
""").df()

print(f"dim_content rows: {len(content_features):,}")
print(content_features['has_been_optimized'].value_counts())

Step 2 (revised): Pull dim_content features with optimization flag
dim_content rows: 411,540
has_been_optimized
False    366144
True      45396
Name: count, dtype: int64


In [21]:
import pandas as pd

feature_frame = daily_agg.merge(
    content_features,
    on=['client_hash_id', 'content_hash_id'],
    how='inner'
)

print(f"Final feature frame rows: {len(feature_frame):,}")
feature_frame.head()

Final feature frame rows: 176,568


,client_hash_id,content_hash_id,total_impressions,avg_position_march,days_with_data,word_count,competition_level,has_been_optimized,is_published
0,client_73cda7b4e4f265ea,content_1e392a54ca96730d,207.0,33.675364,31,<NA>,LOW,False,True
1,client_73cda7b4e4f265ea,content_2e7b4f25a1033e4b,165.0,51.321032,30,<NA>,LOW,False,True
2,client_73cda7b4e4f265ea,content_2bbf031b7abf488a,177.0,49.907002,31,<NA>,LOW,False,True
3,client_73cda7b4e4f265ea,content_368a3b619849acb0,10704.0,5.504164,31,2381,LOW,True,True
4,client_73cda7b4e4f265ea,content_4dd3e1ea3eab7fd0,1154.0,9.736655,31,<NA>,LOW,False,True


## 4. Data limits

*What can this data never tell you? Unbalanced history, GSC-only early rows, window overlaps.*

#### Limit 1: Unbalanced Client History
Not all clients are tracked for a full month. Within 47 distinct clients, 7 clients have very little data for March, and 3 clients joined mid-month and are also not fully covered.
This makes the data of those clients different because they have not acquired enough history depth.

#### Limit 2: GSC-Only Early Rows
While GA4 data is not used in this analysis, clients joining late in the panel have shallow GSC history. A client with only 2 weeks of March 2026 data will have a noisier position-tier baseline than a client with 15 months of history. This means CTR thresholds may be less reliable for newer clients. Consider filtering to clients with minimum gsc_data_start before a certain date.

#### Limit 3: Window Overlap
Using identical windows and labels will negatively impact predictive accuracy. A well-structured approach would involve employing features from one month and designating the following month for the label window.

#### Limit 4: CTR Doesn't Explain Why
CTR cannot exactly identify why a page underperforms. It could be a weak title, SERP features stealing clicks, seasonal patterns, or intent mismatches.
The model only sees aggregate CTR, not query-level data, snippet text, or SERP feature presence. A page flagged as underperforming needs manual review to identify the root cause before the content team invest time in rewrites.
This means the scoring or ranking system is a triage tool, not a solution. It identifies pages worth investigating, but humans must diagnose the actual problem.

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.